# Notebook 5 — Comparación extendida de familias de modelos

**Objetivo**: el Notebook 4 solo comparó Poisson GLM contra LightGBM. Aquí se añaden
**Random Forest** y **XGBoost** (ambos con objetivo/criterio de Poisson, igual que
LightGBM) para comprobar si algún otro modelo de árboles supera al GLM que ganó en el
Notebook 4 — usando exactamente el mismo split temporal, las mismas features y las
mismas métricas (RMSE de goles + LogLoss 1X2 sobre la fase de grupos ya jugada), para
que la comparación sea justa.

Este notebook no sustituye al 4: es un análisis paralelo. Si aquí gana una familia
distinta a la que ya usa el walk-forward del Notebook 4, cambiar `nombre_ganador` allí
es la única modificación necesaria para adoptarla.

Entrada: `data/processed/partidos_features.csv` (Notebook 2).
Salida: `data/processed/comparacion_modelos.csv`.

In [1]:
import json
import time
from pathlib import Path

import lightgbm as lgb
import networkx as nx
import numpy as np
import optuna
import pandas as pd
import xgboost as xgb
from scipy.stats import poisson
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import log_loss, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

optuna.logging.set_verbosity(optuna.logging.WARNING)

DIR_RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DIR_PROCESSED = DIR_RAIZ / "data" / "processed"

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 160)

SEMILLA = 42

# Mismas conclusiones del Notebook 3 que usa el Notebook 4 — se repiten aquí
# porque este notebook se ejecuta de forma independiente, a partir del
# mismo `partidos_features.csv`.
FECHA_CORTE_HISTORICO = pd.Timestamp("1990-01-01")
FECHA_INICIO_MUNDIAL = pd.Timestamp("2026-06-11")
FEATURES_MODELO = [
    "elo_diff", "tendencia_elo_local", "tendencia_elo_visitante",
    "dif_forma_gf_5", "dif_forma_gf_10", "dif_racha_5", "dif_racha_10",
    "dias_descanso_local", "dias_descanso_visitante",
]

df = pd.read_csv(DIR_PROCESSED / "partidos_features.csv", parse_dates=["fecha"])
df = df[df["fecha"] >= FECHA_CORTE_HISTORICO].reset_index(drop=True)
print(f"Partidos desde {FECHA_CORTE_HISTORICO.date()}: {len(df):,}")

Partidos desde 1990-01-01: 32,380


## 5.1 Reconstruir el split de entrenamiento/test (idéntico al Notebook 4)

Mismo criterio que allí: entrenamiento con todo el histórico hasta el día antes del
Mundial, test con la fase de grupos ya jugada — el modelo no ve ni un partido del
torneo antes de que se evalúe contra él.

In [2]:
def detectar_fase_grupos(df_mundial: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Separa los partidos del Mundial 2026 en (fase_de_grupos, eliminatorias),
    igual que en el Notebook 4: camarillas de 4 en el grafo de enfrentamientos
    son grupos reales; el resto es eliminatoria directa."""
    grafo = nx.Graph()
    for _, fila in df_mundial.iterrows():
        grafo.add_edge(fila["equipo_local"], fila["equipo_visitante"])

    camarillas_de_4 = [c for c in nx.enumerate_all_cliques(grafo) if len(c) == 4]
    usados: set[str] = set()
    grupos: list[frozenset[str]] = []
    for c in camarillas_de_4:
        if not (set(c) & usados):
            grupos.append(frozenset(c))
            usados.update(c)

    def _mismo_grupo(fila) -> bool:
        par = {fila["equipo_local"], fila["equipo_visitante"]}
        return any(par <= g for g in grupos)

    es_fase_grupos = df_mundial.apply(_mismo_grupo, axis=1)
    return df_mundial[es_fase_grupos].copy(), df_mundial[~es_fase_grupos].copy()


df_mundial_2026 = df[(df["torneo"] == "FIFA World Cup") & (df["fecha"].dt.year == 2026)]
df_grupos, _df_eliminatoria = detectar_fase_grupos(df_mundial_2026)

df_train_inicial = df[(df["fecha"] < FECHA_INICIO_MUNDIAL) & df["jugado"]].copy()
df_test_grupos = df_grupos[df_grupos["jugado"]].copy()

X_train_inicial = df_train_inicial[FEATURES_MODELO]
X_test_grupos = df_test_grupos[FEATURES_MODELO]

print(f"Entrenamiento: {len(df_train_inicial):,} partidos (hasta {FECHA_INICIO_MUNDIAL.date()})")
print(f"Test (fase de grupos ya jugada): {len(df_test_grupos):,} partidos")

Entrenamiento: 32,286 partidos (hasta 2026-06-11)
Test (fase de grupos ya jugada): 72 partidos


## 5.2 Métrica común: RMSE de goles + LogLoss 1X2

Misma función que en el Notebook 4 (4.5): RMSE sobre los goles exactos y LogLoss sobre
el resultado 1X2 derivado de la rejilla conjunta de Poisson, para que todas las
familias se juzguen con el mismo criterio.

In [3]:
def probabilidades_1x2_desde_lambdas(lam_local: np.ndarray, lam_visitante: np.ndarray,
                                       max_goles: int = 10) -> np.ndarray:
    goles = np.arange(max_goles + 1)
    salida = np.zeros((len(lam_local), 3))
    for i, (la, lv) in enumerate(zip(lam_local, lam_visitante)):
        rejilla = np.outer(poisson.pmf(goles, la), poisson.pmf(goles, lv))
        salida[i] = [np.tril(rejilla, -1).sum(), np.trace(rejilla), np.triu(rejilla, 1).sum()]
    return salida / salida.sum(axis=1, keepdims=True)


def evaluar_modelo_goles(modelos: dict, X: pd.DataFrame, df_partidos: pd.DataFrame, nombre: str) -> dict:
    pred_local = np.clip(modelos["local"].predict(X), 0.01, None)
    pred_visitante = np.clip(modelos["visitante"].predict(X), 0.01, None)

    rmse = np.sqrt(mean_squared_error(
        pd.concat([df_partidos["goles_local"], df_partidos["goles_visitante"]]),
        np.concatenate([pred_local, pred_visitante]),
    ))

    probs = probabilidades_1x2_desde_lambdas(pred_local, pred_visitante)
    y_real = df_partidos["resultado_1x2"].map({"LOCAL": 0, "EMPATE": 1, "VISITANTE": 2}).values
    logloss = log_loss(y_real, probs, labels=[0, 1, 2])

    print(f"[{nombre}] RMSE goles: {rmse:.4f}  |  LogLoss 1X2: {logloss:.4f}")
    return {"rmse": rmse, "logloss": logloss}

## 5.3 Búsqueda de hiperparámetros, común a las tres familias de árboles/boosting

Una sola función de CV temporal (`TimeSeriesSplit`, nunca K-Fold barajado) y un solo
bucle de Optuna sirven para LightGBM, XGBoost y Random Forest: cada familia solo aporta
su propio "constructor" (`trial -> estimador sin entrenar`), reutilizado también para
reconstruir el modelo final con `optuna.trial.FixedTrial` una vez conocidos los mejores
hiperparámetros — así no hay que mantener el mapeo hiperparámetros -> estimador por
duplicado.

In [4]:
RUTA_PROGRESO = DIR_PROCESSED / "_progreso_notebook05.json"


def rmse_cv_temporal(estimador, X: pd.DataFrame, y: pd.Series, n_splits: int = 5) -> float:
    cv = TimeSeriesSplit(n_splits=n_splits)
    rmses = []
    for idx_train, idx_val in cv.split(X):
        modelo = clone(estimador).fit(X.iloc[idx_train], y.iloc[idx_train])
        pred = np.clip(modelo.predict(X.iloc[idx_val]), 0, None)
        rmses.append(np.sqrt(mean_squared_error(y.iloc[idx_val], pred)))
    return float(np.mean(rmses))


def optimizar_familia(construir_estimador, X: pd.DataFrame, y: pd.Series, n_trials: int = 40, nombre: str = "") -> optuna.Study:
    inicio = time.time()

    def _callback(estudio: optuna.Study, trial: optuna.trial.FrozenTrial) -> None:
        # nbconvert no vuelca nada a stdout hasta que el notebook entero
        # termina -- por eso el progreso se escribe aparte, a un fichero en
        # disco, consultable en cualquier momento aunque el notebook siga
        # corriendo (`cat data/processed/_progreso_notebook05.json`).
        progreso = {
            "familia_target": nombre,
            "trial": trial.number + 1,
            "n_trials": n_trials,
            "segundos_transcurridos": round(time.time() - inicio, 1),
            "mejor_rmse_hasta_ahora": estudio.best_value,
        }
        RUTA_PROGRESO.write_text(json.dumps(progreso, indent=2))

    estudio = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEMILLA))
    estudio.optimize(lambda trial: rmse_cv_temporal(construir_estimador(trial), X, y),
                      n_trials=n_trials, show_progress_bar=False, callbacks=[_callback])
    print(f"[{nombre}] Mejor RMSE (CV temporal): {estudio.best_value:.4f} en {time.time() - inicio:.0f}s "
          f"| hiperparámetros: {estudio.best_params}")
    return estudio

## 5.4 Espacio de hiperparámetros por familia

Mismo espacio de LightGBM que en el Notebook 4 (para que su resultado aquí sea
comparable), y equivalentes razonables para XGBoost (`count:poisson`) y Random Forest
(`criterion="poisson"`, disponible en scikit-learn desde la 1.0).

In [5]:
def construir_lightgbm(trial: optuna.Trial) -> lgb.LGBMRegressor:
    return lgb.LGBMRegressor(
        objective="poisson", random_state=SEMILLA, verbosity=-1,
        n_estimators=trial.suggest_int("n_estimators", 100, 300),
        num_leaves=trial.suggest_int("num_leaves", 8, 64),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        min_child_samples=trial.suggest_int("min_child_samples", 5, 100),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
    )


def construir_xgboost(trial: optuna.Trial) -> xgb.XGBRegressor:
    return xgb.XGBRegressor(
        objective="count:poisson", random_state=SEMILLA, verbosity=0,
        n_estimators=trial.suggest_int("n_estimators", 100, 300),
        max_depth=trial.suggest_int("max_depth", 2, 8),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 20),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
    )


def construir_random_forest(trial: optuna.Trial) -> RandomForestRegressor:
    # Rangos deliberadamente más pequeños que los de LightGBM/XGBoost: un árbol
    # de Random Forest sin gradient boosting ni histogramas es mucho más lento
    # de entrenar por árbol, y 500 árboles a profundidad 20 en 32k filas x 40
    # trials x 5 folds x 2 targets llegó a tardar más de 3 horas en la primera
    # ejecución -- no aporta rigor extra que justifique ese coste aquí.
    return RandomForestRegressor(
        criterion="poisson", random_state=SEMILLA, n_jobs=-1,
        n_estimators=trial.suggest_int("n_estimators", 50, 150),
        max_depth=trial.suggest_int("max_depth", 3, 10),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 3, 50),
        max_features=trial.suggest_float("max_features", 0.3, 1.0),
    )


FAMILIAS_ARBOLES = {
    "lightgbm": construir_lightgbm,
    "xgboost": construir_xgboost,
    "random_forest": construir_random_forest,
}

# Segunda vuelta tras un primer intento que superó las 3 horas: rangos más
# pequeños en las tres familias (arriba) y menos trials en todas, no solo en
# Random Forest -- prioriza terminar en un tiempo razonable sobre exprimir el
# último decimal de un análisis que es secundario (el Notebook 4 ya decide
# la familia que de verdad se usa).
TRIALS_POR_FAMILIA = {"lightgbm": 15, "xgboost": 15, "random_forest": 15}

## 5.5 Afinar cada familia y entrenar el modelo final

El GLM de Poisson (Notebook 4, sección 4.3) entra como referencia sin afinar — ya se
demostró allí que Optuna no le aporta nada porque no tiene hiperparámetros que buscar
más allá de la regularización L2 fija.

In [6]:
def entrenar_poisson_glm(X: pd.DataFrame, y: pd.Series, alpha: float = 0.5):
    return make_pipeline(StandardScaler(), PoissonRegressor(alpha=alpha, max_iter=1000)).fit(X, y)


modelos_por_familia = {
    "glm": {
        "local": entrenar_poisson_glm(X_train_inicial, df_train_inicial["goles_local"]),
        "visitante": entrenar_poisson_glm(X_train_inicial, df_train_inicial["goles_visitante"]),
    },
}

for nombre_familia, construir in FAMILIAS_ARBOLES.items():
    n_trials = TRIALS_POR_FAMILIA[nombre_familia]
    estudio_local = optimizar_familia(construir, X_train_inicial, df_train_inicial["goles_local"], n_trials=n_trials, nombre=f"{nombre_familia}/local")
    estudio_visitante = optimizar_familia(construir, X_train_inicial, df_train_inicial["goles_visitante"], n_trials=n_trials, nombre=f"{nombre_familia}/visitante")

    modelos_por_familia[nombre_familia] = {
        "local": construir(optuna.trial.FixedTrial(estudio_local.best_params)).fit(X_train_inicial, df_train_inicial["goles_local"]),
        "visitante": construir(optuna.trial.FixedTrial(estudio_visitante.best_params)).fit(X_train_inicial, df_train_inicial["goles_visitante"]),
    }

[lightgbm/local] Mejor RMSE (CV temporal): 1.5926 en 16s | hiperparámetros: {'n_estimators': 198, 'num_leaves': 9, 'learning_rate': 0.048429731307292646, 'min_child_samples': 12, 'reg_lambda': 0.3702923527973761, 'subsample': 0.8729665514595508, 'colsample_bytree': 0.9981098783963183}


[lightgbm/visitante] Mejor RMSE (CV temporal): 1.2672 en 17s | hiperparámetros: {'n_estimators': 225, 'num_leaves': 10, 'learning_rate': 0.03939913177564375, 'min_child_samples': 35, 'reg_lambda': 0.0012294606554986399, 'subsample': 0.9747495761264378, 'colsample_bytree': 0.8965519941223506}


[xgboost/local] Mejor RMSE (CV temporal): 1.5931 en 11s | hiperparámetros: {'n_estimators': 295, 'max_depth': 4, 'learning_rate': 0.035452943469397764, 'min_child_weight': 7, 'reg_lambda': 0.23079754987887754, 'subsample': 0.881307350083289, 'colsample_bytree': 0.8846069148170447}


[xgboost/visitante] Mejor RMSE (CV temporal): 1.2671 en 11s | hiperparámetros: {'n_estimators': 295, 'max_depth': 4, 'learning_rate': 0.035452943469397764, 'min_child_weight': 7, 'reg_lambda': 0.23079754987887754, 'subsample': 0.881307350083289, 'colsample_bytree': 0.8846069148170447}


[random_forest/local] Mejor RMSE (CV temporal): 1.5935 en 1419s | hiperparámetros: {'n_estimators': 96, 'max_depth': 9, 'min_samples_leaf': 12, 'max_features': 0.6599641068895281}


[random_forest/visitante] Mejor RMSE (CV temporal): 1.2683 en 1642s | hiperparámetros: {'n_estimators': 146, 'max_depth': 7, 'min_samples_leaf': 23, 'max_features': 0.31257331293037366}


## 5.6 Comparación en fase de grupos

Mismo test que el Notebook 4 (72 partidos de fase de grupos, evaluados "en frío"): la
familia ganadora es la de menor RMSE de goles.

In [7]:
resultados = []
for nombre_familia, modelos in modelos_por_familia.items():
    metricas = evaluar_modelo_goles(modelos, X_test_grupos, df_test_grupos, nombre_familia)
    resultados.append({"familia": nombre_familia, **metricas})

df_comparacion = pd.DataFrame(resultados).sort_values("rmse").reset_index(drop=True)
print("\n=== Comparación final (ordenada por RMSE) ===")
print(df_comparacion.round(4).to_string(index=False))

ganador = df_comparacion.iloc[0]["familia"]
print(f"\nFamilia ganadora: {ganador}")

ruta_comparacion = DIR_PROCESSED / "comparacion_modelos.csv"
df_comparacion.to_csv(ruta_comparacion, index=False)
print(f"Comparación guardada en {ruta_comparacion}")

[glm] RMSE goles: 1.2624  |  LogLoss 1X2: 0.9020
[lightgbm] RMSE goles: 1.3157  |  LogLoss 1X2: 0.9140
[xgboost] RMSE goles: 1.2925  |  LogLoss 1X2: 0.9068
[random_forest] RMSE goles: 1.2788  |  LogLoss 1X2: 0.9042

=== Comparación final (ordenada por RMSE) ===
      familia   rmse  logloss
          glm 1.2624   0.9020
random_forest 1.2788   0.9042
      xgboost 1.2925   0.9068
     lightgbm 1.3157   0.9140

Familia ganadora: glm
Comparación guardada en /Users/danielcanteragomez/portfolio/wc-2026-match-predictor/data/processed/comparacion_modelos.csv


## Resumen

- Se compararon 4 familias sobre el mismo split y las mismas métricas que el Notebook 4:
  Poisson GLM (sin afinar), LightGBM, XGBoost y Random Forest (estas tres últimas
  afinadas con Optuna sobre CV temporal).
- El resultado queda en `data/processed/comparacion_modelos.csv`, ordenado por RMSE.
- Si la familia ganadora aquí no es `"glm"` (la que usa actualmente el walk-forward del
  Notebook 4), basta con cambiar el valor de `nombre_ganador` en el Notebook 4 y
  reejecutarlo — `reentrenar_modelos` ahí solo sabe construir `"glm"` o `"lgbm_optuna"`
  hoy, así que adoptar Random Forest o XGBoost en el propio walk-forward requeriría
  además añadir esa rama.